# Stage 1 — Raw Page Structure Inspection


## 1. Priority Classification

In [48]:
import json
import re
from pathlib import Path
from collections import defaultdict

RAW_DIR = Path('../../data/raw_scraped_pages')
CLASSIF = Path('../../docs/url_classification.json')

# --- load pages (page_*.json only; footer.json is handled separately in §2) ---
# raw files have two schemas:
#   schema A (newer spider): page_url / page_title / all_text_markdown / sections
#   schema B (older spider): source_url / text_markdown / links / raw_html
def get_url(page):
    return page.get('page_url') or page.get('source_url', '')

def get_title(page):
    return page.get('page_title', '')

pages = []
schema_counts = {'A': 0, 'B': 0}
for fpath in sorted(RAW_DIR.glob('page_*.json')):
    with open(fpath, encoding='utf-8') as f:
        p = json.load(f)
    pages.append(p)
    if 'page_url' in p:
        schema_counts['A'] += 1
    else:
        schema_counts['B'] += 1

print(f'Loaded {len(pages)} raw pages  (page_*.json only, footer.json excluded)')
print(f'  Schema A (page_url / page_title / all_text_markdown): {schema_counts["A"]}')
print(f'  Schema B (source_url / text_markdown / raw_html):     {schema_counts["B"]}')

# --- load classification ---
with open(CLASSIF, encoding='utf-8') as f:
    classif = json.load(f)

entries      = classif['entries']
url_patterns = classif['url_patterns']
print(f'\nClassification loaded: {len(entries)} entries, {len(url_patterns)} url_patterns')

Loaded 176 raw pages  (page_*.json only, footer.json excluded)
  Schema A (page_url / page_title / all_text_markdown): 176
  Schema B (source_url / text_markdown / raw_html):     0

Classification loaded: 68 entries, 19 url_patterns


In [49]:
# --- URL normalizer: raw pages may lack trailing slash ---
def norm(url: str) -> str:
    return url.rstrip('/')

# Build lookup: norm(url) -> entry
entry_lookup = {}
for e in entries:
    entry_lookup[norm(e['url'])] = e
    entry_lookup.setdefault(norm(e['canonical_url']), e)

def classify_url(url: str) -> dict:
    key = norm(url)
    if key in entry_lookup:
        return entry_lookup[key]
    for pat in url_patterns:
        if re.match(pat['regex'], url):
            return pat
    return {'priority': 'skip', 'url': url}

# --- classify every page ---
PRIORITY_ORDER = ['must', 'dsi-general', 'optional', 'alias', 'skip']
classified = defaultdict(list)   # priority -> list of dicts

for p in pages:
    url   = get_url(p)
    title = get_title(p)
    result   = classify_url(url)
    priority = result.get('priority', 'unlisted')
    classified[priority].append({'url': url, 'title': title})

print('Classification summary:')
for pri in PRIORITY_ORDER:
    print(f'  {pri:12s}: {len(classified[pri]):3d} pages')

Classification summary:
  must        :  14 pages
  dsi-general :   7 pages
  optional    :  10 pages
  alias       :   0 pages
  skip        : 145 pages


In [50]:
# --- Print each priority group in full ---
for priority in PRIORITY_ORDER:
    group = classified[priority]
    if not group:
        continue
    print(f'\n{"="*72}')
    print(f'  [{priority.upper()}]  ({len(group)} pages)')
    print(f'{"="*72}')
    for item in sorted(group, key=lambda x: x['url']):
        print(f'  {item["url"]}')


  [MUST]  (14 pages)
  https://datascience.uchicago.edu/education/masters-programs/in-person-program
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-project-archive
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/capstone-projects
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/career-outcomes
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/course-progressions
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/events-deadlines
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/faqs
  https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/how-to-apply
  https://datascience.uchicago.edu/education/masters-programs/ms-in-appl

In [51]:
# --- Gap check: sitemap must/should entries MISSING from raw ---
raw_url_norms = {norm(get_url(p)) for p in pages}

print('Sitemap must/should entries MISSING from raw_scraped_pages:\n')
missing = []
for e in entries:
    if e['priority'] in ('must', 'should'):
        if norm(e['url']) not in raw_url_norms and norm(e['canonical_url']) not in raw_url_norms:
            missing.append(e)

if missing:
    for e in missing:
        print(f'  [{e["priority"]}]  {e["url"]}')
else:
    print('  None — all must/should entries are present in raw data.')

Sitemap must/should entries MISSING from raw_scraped_pages:

  None — all must/should entries are present in raw data.


## 2. Header & Footer Classification

In [52]:
FOOTER_PATH = RAW_DIR.parent / 'raw_scraped_pages' / 'footer.json'

with open(FOOTER_PATH, encoding='utf-8') as f:
    footer = json.load(f)

footer_hrefs = [link['href'] for link in footer.get('links', []) if link.get('href')]
print(f'Footer hrefs: {len(footer_hrefs)} total')

# Compare against the 177 raw scraped pages
raw_url_norms = {norm(get_url(p)) for p in pages}

footer_classified = []
for href in footer_hrefs:
    tag = 'dsi-general' if norm(href) in raw_url_norms else 'footer-only'
    footer_classified.append({'url': href, 'tag': tag})

dsi_general_count = sum(1 for x in footer_classified if x['tag'] == 'dsi-general')
footer_only_count  = sum(1 for x in footer_classified if x['tag'] == 'footer-only')

print(f'  In raw pages  → dsi-general : {dsi_general_count}')
print(f'  Not in raw    → footer-only : {footer_only_count}')
print()
for item in sorted(footer_classified, key=lambda x: x['url']):
    print(f'  [{item["tag"]:12s}]  {item["url"]}')

Footer hrefs: 21 total
  In raw pages  → dsi-general : 12
  Not in raw    → footer-only : 9

  [footer-only ]  https://accessibility.uchicago.edu
  [footer-only ]  https://bsky.app/profile/dsi-uchicago.bsky.social
  [dsi-general ]  https://datascience.uchicago.edu
  [dsi-general ]  https://datascience.uchicago.edu/about
  [dsi-general ]  https://datascience.uchicago.edu/about/contact
  [dsi-general ]  https://datascience.uchicago.edu/about/jobs
  [dsi-general ]  https://datascience.uchicago.edu/about/leadership-staff
  [dsi-general ]  https://datascience.uchicago.edu/education
  [dsi-general ]  https://datascience.uchicago.edu/news-events/events
  [dsi-general ]  https://datascience.uchicago.edu/news-events/news
  [dsi-general ]  https://datascience.uchicago.edu/newsletter-archive
  [dsi-general ]  https://datascience.uchicago.edu/nondiscrimination-statement
  [dsi-general ]  https://datascience.uchicago.edu/outreach
  [dsi-general ]  https://datascience.uchicago.edu/research
  [footer

## 3. Generate Bottom Table

### 3.1  Build url_classes from classified (§1) + footer_classified (§2)

In [53]:
# Base: classified['must/dsi-general/optional/skip']  (§1 result)
# Promote: footer hrefs that ARE in raw pages (tag='dsi-general' in §2)
#          → if currently skip/not in top-3, move to dsi-general
# Add:     footer hrefs NOT in raw pages (tag='footer-only' in §2)
#          → footer-only bucket

# Build norm→priority map from §1 classified
_norm_to_pri = {}
for pri in ('must', 'dsi-general', 'optional', 'skip'):
    for item in classified[pri]:
        _norm_to_pri[norm(item['url'])] = pri

url_classes = {
    'must':        [item['url'] for item in classified['must']],
    'dsi-general': [item['url'] for item in classified['dsi-general']],
    'optional':    [item['url'] for item in classified['optional']],
    'footer-only': [],
}

for fc in footer_classified:
    n   = norm(fc['url'])
    tag = fc['tag']   # 'dsi-general' = in raw pages | 'footer-only' = not in raw
    if tag == 'footer-only':
        url_classes['footer-only'].append(fc['url'])
    else:
        # in raw pages — promote to dsi-general if currently skip or unclassified
        current = _norm_to_pri.get(n, 'skip')
        if current not in ('must', 'dsi-general', 'optional'):
            url_classes['dsi-general'].append(fc['url'])

# ── Print counts ──
print('url_classes counts:')
for pri in ('must', 'dsi-general', 'optional', 'footer-only'):
    print(f'  {pri:12s}: {len(url_classes[pri])}')

print('\ndsi-general URLs:')
for u in sorted(url_classes['dsi-general']):
    print(f'  {u}')

print('\nfooter-only URLs:')
for u in sorted(url_classes['footer-only']):
    print(f'  {u}')

url_classes counts:
  must        : 14
  dsi-general : 14
  optional    : 10
  footer-only : 9

dsi-general URLs:
  https://datascience.uchicago.edu
  https://datascience.uchicago.edu/about
  https://datascience.uchicago.edu/about/about-dsi
  https://datascience.uchicago.edu/about/contact
  https://datascience.uchicago.edu/about/jobs
  https://datascience.uchicago.edu/about/leadership-staff
  https://datascience.uchicago.edu/education
  https://datascience.uchicago.edu/insights
  https://datascience.uchicago.edu/news-events/events
  https://datascience.uchicago.edu/news-events/news
  https://datascience.uchicago.edu/newsletter-archive
  https://datascience.uchicago.edu/nondiscrimination-statement
  https://datascience.uchicago.edu/outreach
  https://datascience.uchicago.edu/research

footer-only URLs:
  https://accessibility.uchicago.edu
  https://bsky.app/profile/dsi-uchicago.bsky.social
  https://github.com/uchicago-dsi
  https://instagram.com/dsi_uchicago
  https://physicalsciences.

### 3.2  Build records (must/dsi-general/optional) + footer dict

In [54]:
KEEP_PRIORITIES = {'must', 'dsi-general', 'optional'}

# norm(url) → priority from url_classes
_cls_norm_to_pri = {}
for pri, urls in url_classes.items():
    for u in urls:
        _cls_norm_to_pri[norm(u)] = pri

# Build norm(url) → filename in one pass (page_*.json only)
_norm_to_file = {}
for fpath in sorted(RAW_DIR.glob('page_*.json')):
    with open(fpath, encoding='utf-8') as f:
        _p = json.load(f)
    _norm_to_file[norm(get_url(_p))] = fpath.name

records = []
for p in pages:
    url      = get_url(p)
    priority = _cls_norm_to_pri.get(norm(url))
    if priority not in KEEP_PRIORITIES:
        continue
    entry = entry_lookup.get(norm(url), {})
    records.append({
        'source_file':       _norm_to_file.get(norm(url), ''),
        'url':               url,
        'canonical_url':     entry.get('canonical_url', url),
        'priority':          priority,
        'depth':             entry.get('depth', None),
        'special_handling':  entry.get('special_handling', []),
        'page_title':        p.get('page_title', ''),
        'meta_description':  p.get('meta_description', ''),
        'breadcrumbs':       p.get('breadcrumbs', []),
        'scraped_at':        p.get('scraped_at', ''),
        'all_text_markdown': p.get('all_text_markdown') or p.get('text_markdown', ''),
        'sections':          p.get('sections', None),
    })


# ── Summary ──
print(f'records total: {len(records)}')
pri_counts = defaultdict(int)
for r in records:
    pri_counts[r['priority']] += 1
for pri in ('must', 'dsi-general', 'optional'):
    print(f'  {pri:12s}: {pri_counts[pri]}')

tag_counts = defaultdict(int)
for r in records:
    for tag in r['special_handling']:
        tag_counts[tag] += 1
print('\nspecial_handling tag counts:')
if tag_counts:
    for tag, cnt in sorted(tag_counts.items(), key=lambda x: -x[1]):
        files = [r['source_file'] for r in records if tag in r['special_handling']]
        print(f'  {tag:45s}: {cnt}  ({", ".join(files)})')
else:
    print('  (none)')



records total: 38
  must        : 14
  dsi-general : 14
  optional    : 10

special_handling tag counts:
  verify_accordion_expansion                   : 3  (page_146.json, page_148.json, page_159.json)
  flag_for_periodic_rescrape                   : 1  (page_164.json)
  exclude_registration_links                   : 1  (page_164.json)
  exclude_image_embedded_content               : 1  (page_165.json)


In [61]:
from urllib.parse import urlparse

def _platform_name(href: str) -> str:
    """Derive platform name from hostname when link text is empty.
    https://x.com/...          -> 'X'
    https://www.linkedin.com/  -> 'LinkedIn'
    https://bsky.app/...       -> 'Bsky'
    """
    host = urlparse(href).hostname or ''
    host = host.removeprefix('www.')   # strip leading www.
    name = host.split('.')[0]          # take part before first dot
    return name.capitalize()

# ── footer dict: links from footer.json ──
footer_links = [
    {
        'text': link.get('text') or _platform_name(link['href']),
        'href': link['href'],
    }
    for link in footer.get('links', [])
    if link.get('href')
]

print(f'footer_links: {len(footer_links)} entries')
for fl in footer_links:
    print(f'  {fl["text"]!r:30s}  {fl["href"]}')

footer_links: 21 entries
  'DSI'                           https://datascience.uchicago.edu
  'About'                         https://datascience.uchicago.edu/about
  'People'                        https://datascience.uchicago.edu/about/leadership-staff
  'Research'                      https://datascience.uchicago.edu/research
  'Education'                     https://datascience.uchicago.edu/education
  'Outreach'                      https://datascience.uchicago.edu/outreach
  'News'                          https://datascience.uchicago.edu/news-events/news
  'Events'                        https://datascience.uchicago.edu/news-events/events
  'Jobs'                          https://datascience.uchicago.edu/about/jobs
  'Newsletter Archive'            https://datascience.uchicago.edu/newsletter-archive
  'Physical Sciences Division'    https://physicalsciences.uchicago.edu
  'Nondiscrimination Statement'   https://datascience.uchicago.edu/nondiscrimination-statement
  'Accessibilit